In [ ]:
data_json = [{
    "title": "AIREST-CLINICAL Grant Proposal CLinical Part",
    "file_path": "data/Draft_Tpl_Information-on-clinical-studies_(HE EIC UKRAINIANTECH)-2025.rtf (1).docx",
    "description": 'The document is a grant proposal annex detailing the rationale for the **AIREST-CLINICAL** investigation, a prospective, observational study to validate an AI-driven multimodal digital biomarker platform (using voice, facial expression, etc.) for diagnosing Post-Traumatic Stress Disorder (PTSD), Generalized Anxiety Disorder (GAD), and depressive disorders. The study is critically important to address the severe mental health crisis in Ukraine, which is characterized by a high prevalence of these conditions, a vast diagnostic gap (154:1 to 224:1), and a severely limited psychiatric workforce. The AIREST solution is projected to provide a sixfold acceleration in diagnostic throughput, reducing the time from symptom onset to diagnosis from 12–24 months to 2–6 months, thereby enabling early intervention, preventing an estimated 5,000–8,750 suicides per year, and generating significant long-term economic and workforce preservation benefits, especially since current diagnostic methods are subjective and existing digital biomarker studies are small-scale and do not focus on this war-affected population.',
    "keywords": [
        "AIREST-CLINICAL",
        "grant proposal",
        "digital biomarker",
        "AI-driven diagnosis",
        "Post-Traumatic Stress Disorder (PTSD)",
        "Generalized Anxiety Disorder (GAD)",
        "depressive disorders",
        "mental health crisis",
        "Ukraine",
        "psychiatric workforce",
        "diagnostic throughput",
        "early intervention",
        "Multimodal Machine Learning (MML)",
        'Depression',
        'Mental Health Crisis',
        'War-affected population',
        'Diagnostic gap',
        'Psychiatric workforce shortage',
        'Early intervention',
        "Horizon Europe (HE)"]
},
{
    "title": "AIREST-CLINICAL Grant Proposal Technical and Business Part",
    "file_path": "data/DRAFT_Tpl_Application_Form_Part_B_HE_EIC_UKRAINE_2025_AIREST_5.docx",
    "description": "The document is an application for the AIREST project, which aims to develop and validate a browser-based, multimodal self-assessment tool for scalable mental health screening and monitoring, particularly for war-affected, Ukrainian-language populations. The project addresses the critical diagnostic gap in Ukraine, where millions of adults meet criteria for disorders like PTSD but few receive a formal diagnosis due to a limited psychiatric workforce and the long, subjective nature of traditional 60-90 minute interviews. AIREST's core scientific novelty is an objective, 10-15 minute protocol that captures and time-aligns multiple digital biomarkers (vocal, facial expression, attention-bias) and processes them into interpretable clinical indices. The objectives are to advance the system from a TRL-4 prototype to a TRL 6-7 clinically validated Minimum Viable Product (MVP) via a large-scale (N=1,375) clinical investigation, while ensuring regulatory compliance as a Class IIa Software as a Medical Device (SaMD) under EU MDR and alignment with the EU AI Act. The projected impact includes reducing the diagnostic burden by 83%, freeing up millions of clinician hours for therapy, preventing suicide and severe deterioration, and yielding significant economic savings for European health systems.",
    "keywords": [
        "AIREST-TECHNICAL",
        "technical annex",
        "AI-driven diagnosis",
        "Multimodal Digital Biomarkers",
        "Multimodal Machine Learning (MML)",
        "data collection",
        "study design",
        "data processing",
        "validation protocols",
        "data security",
        "privacy considerations",
        "ethical standards",
        "Ukraine",
        "mental health disorders",
        "Horizon Europe (HE)",
        "SaMD (Software as a Medical Device)",
        "EU MDR", "EU AI Act",
        "TRL 4-7", "Investment Readiness",
        "Deep-Tech", "Diagnostic Gap",
        "Objective Assessment" 
    ]
}
]



In [ ]:
import json
from pathlib import Path

OUT_PATH = Path(__file__).resolve().parents[1] / "data" / "data.json"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(data_json, f, ensure_ascii=False, indent=2)
print(f"Wrote JSON with {len(data_json)} items to {OUT_PATH}")

In [ ]:
import json
with open("data/data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Minimal Graph visualization

In [4]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, List, Dict, Any, Optional

from neo4j import GraphDatabase


@dataclass
class DocumentRecord:
    """
    In-memory representation of one document entry.

    Matches your JSON:
    {
        "title": str,
        "file_path": str,
        "description": str,
        "keywords": list[str]
    }
    """
    title: str
    file_path: str
    description: str
    keywords: List[str]

    @property
    def file_name(self) -> str:
        return Path(self.file_path).name


class Neo4jKnowledgeBase:
    """
    Minimal graph layer for your local documents.

    Creates:
      - (:Document {file_path, title, description, file_name})
      - (:Keyword {name})
      - (Document)-[:HAS_KEYWORD]->(Keyword)
      - (Document)-[:SIMILAR_TO {common_keywords}]->(Document)
    """

    def __init__(
        self,
        uri: str,
        user: str,
        password: str,
        database: str = "neo4j",
    ) -> None:
        """
        :param uri: Neo4j URI, e.g. 'neo4j://localhost:7687'
        :param user: Neo4j username
        :param password: Neo4j password
        :param database: Database name (default: 'neo4j')
        """
        self._driver = GraphDatabase.driver(uri, auth=(user, password))
        self._database = database

    # -------- lifecycle --------

    def close(self) -> None:
        self._driver.close()

    # -------- internal helpers (transactions) --------

    def _execute_write(self, query: str, **params):
        """Run a write query inside a managed transaction."""
        def _tx(tx):
            result = tx.run(query, **params)
            return list(result)  # helps debugging if you ever need rows

        with self._driver.session(database=self._database) as session:
            return session.execute_write(_tx)

    def _execute_read(self, query: str, **params):
        """Run a read query inside a managed transaction."""
        def _tx(tx):
            result = tx.run(query, **params)
            return list(result)

        with self._driver.session(database=self._database) as session:
            return session.execute_read(_tx)

    # -------- schema / constraints --------

    def setup_schema(self) -> None:
        """
        Create idempotent constraints for Document and Keyword.
        """
        # Unique Document by file_path (fits your local data nicely)
        self._execute_write(
            """
            CREATE CONSTRAINT document_file_path IF NOT EXISTS
            FOR (d:Document)
            REQUIRE d.file_path IS UNIQUE
            """
        )

        # Unique Keyword by name
        self._execute_write(
            """
            CREATE CONSTRAINT keyword_name IF NOT EXISTS
            FOR (k:Keyword)
            REQUIRE k.name IS UNIQUE
            """
        )

    # -------- ingestion API --------

    def ingest_data_json(self, data_json: List[Dict[str, Any]]) -> None:
        """
        High-level method: takes your list[dict] and ingests everything.
        """
        docs = [self._from_dict(d) for d in data_json]

        for doc in docs:
            self.upsert_document(doc)
            self.attach_keywords(doc)

    def _from_dict(self, d: Dict[str, Any]) -> DocumentRecord:
        """
        Convert raw dict (from your JSON) → DocumentRecord with basic validation.
        """
        title = (d.get("title") or "").strip()
        file_path = (d.get("file_path") or "").strip()
        description = d.get("description") or ""
        keywords_raw = d.get("keywords") or []

        if not isinstance(keywords_raw, list):
            raise ValueError("`keywords` must be a list[str]")

        # Normalize keywords: strip, lower, drop empties, deduplicate
        normalized_keywords: List[str] = []
        for kw in keywords_raw:
            if kw is None:
                continue
            s = str(kw).strip()
            if not s:
                continue
            s_lower = s.lower()
            if s_lower not in normalized_keywords:
                normalized_keywords.append(s_lower)

        if not file_path:
            raise ValueError("`file_path` must be non-empty")

        return DocumentRecord(
            title=title or Path(file_path).name,
            file_path=file_path,
            description=description,
            keywords=normalized_keywords,
        )

    # -------- document / keyword operations --------

    def upsert_document(self, doc: DocumentRecord) -> None:
        """
        Create or update a :Document node from DocumentRecord.
        """
        self._execute_write(
            """
            MERGE (d:Document {file_path: $file_path})
            SET d.title         = $title,
                d.description   = $description,
                d.file_name     = $file_name,
                d.ingested_at   = datetime()
            """,
            file_path=doc.file_path,
            title=doc.title,
            description=doc.description,
            file_name=doc.file_name,
        )

    def attach_keywords(self, doc: DocumentRecord) -> None:
        """
        Ensure :Keyword nodes exist and attach them via :HAS_KEYWORD.
        """
        if not doc.keywords:
            return

        self._execute_write(
            """
            MATCH (d:Document {file_path: $file_path})
            UNWIND $keywords AS kw_name
            MERGE (k:Keyword {name: kw_name})
            MERGE (d)-[:HAS_KEYWORD]->(k)
            """,
            file_path=doc.file_path,
            keywords=doc.keywords,
        )

    def build_similarity_edges(self, min_common_keywords: int = 1) -> None:
        """
        Create SIMILAR_TO relationships between documents that share keywords.

        Relationship:
          (d1:Document)-[r:SIMILAR_TO]-(d2:Document)
        with property r.common_keywords
        """
        self._execute_write(
            """
            MATCH (d1:Document)-[:HAS_KEYWORD]->(k:Keyword)<-[:HAS_KEYWORD]-(d2:Document)
            WHERE id(d1) < id(d2)   // avoid duplicates & self-pairs
            WITH d1, d2, count(DISTINCT k) AS commonKeywords
            WHERE commonKeywords >= $min_common_keywords
            MERGE (d1)-[r:SIMILAR_TO]-(d2)
            SET r.common_keywords = commonKeywords,
                r.created_at      = datetime()
            """,
            min_common_keywords=min_common_keywords,
        )

    # -------- convenience read helpers (for debugging & basic "visualization" from Python) --------

    def list_documents(self, limit: int = 50):
        """
        Returns basic info + keywords for first N documents.
        """
        return self._execute_read(
            """
            MATCH (d:Document)
            OPTIONAL MATCH (d)-[:HAS_KEYWORD]->(k:Keyword)
            RETURN d.file_path AS file_path,
                   d.title AS title,
                   collect(DISTINCT k.name) AS keywords
            ORDER BY title
            LIMIT $limit
            """,
            limit=limit,
        )

    def similar_documents(self, file_path: str, limit: int = 20):
        """
        Returns documents connected to the given file via SIMILAR_TO.
        """
        return self._execute_read(
            """
            MATCH (d:Document {file_path: $file_path})-[r:SIMILAR_TO]-(other:Document)
            RETURN other.file_path AS other_file_path,
                   other.title     AS title,
                   r.common_keywords AS common_keywords
            ORDER BY r.common_keywords DESC
            LIMIT $limit
            """,
            file_path=file_path,
        )


In [8]:
kb = Neo4jKnowledgeBase(
        uri="neo4j://localhost:7687",
        user="neo4j",
        password="airestairest",
        database="neo4j",
    )

In [9]:
kb.setup_schema()

In [10]:
kb.ingest_data_json(data_json)

In [11]:
kb.build_similarity_edges(min_common_keywords=1)

Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=3, column=19, offset=109>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 109, 'line': 3, 'column': 19}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            MATCH (d1:Document)-[:HAS_KEYWORD]->(k:Keyword)<-[:HAS_KEYWORD]-(d2:Document)\n            WHERE id(d1) < id(d2)   // avoid duplicates & self-pairs\n            WITH d1, d2, count(DISTINCT k) AS commonKeywords\n            WHERE commonKeywords >= $min_common_keywords\n            MERGE (d1)-[r:SIMILAR_TO]-(d2)\n            SET

In [12]:
rows = kb.list_documents(limit=10)

In [13]:
for row in rows:
    print(row)

<Record file_path='data/Draft_Tpl_Information-on-clinical-studies_(HE EIC UKRAINIANTECH)-2025.rtf (1).docx' title='AIREST-CLINICAL Grant Proposal CLinical Part' keywords=['horizon europe (he)', 'psychiatric workforce shortage', 'diagnostic gap', 'war-affected population', 'depression', 'multimodal machine learning (mml)', 'early intervention', 'diagnostic throughput', 'psychiatric workforce', 'ukraine', 'mental health crisis', 'depressive disorders', 'generalized anxiety disorder (gad)', 'post-traumatic stress disorder (ptsd)', 'ai-driven diagnosis', 'digital biomarker', 'grant proposal', 'airest-clinical']>
<Record file_path='data/DRAFT_Tpl_Application_Form_Part_B_HE_EIC_UKRAINE_2025_AIREST_5.docx' title='AIREST-CLINICAL Grant Proposal Technical and Business Part' keywords=['diagnostic gap', 'trl 4-7', 'deep-tech', 'eu mdr', 'objective assessment', 'samd (software as a medical device)', 'investment readiness', 'eu ai act', 'horizon europe (he)', 'mental health disorders', 'ukraine', '

In [ ]:
kb.close()

In [17]:
!uv pip install langchain langchain-openai langchain-neo4j neo4j

Audited 4 packages in 35ms


In [18]:
from langchain_openai import ChatOpenAI
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain

# 1) LLM из LM Studio
llm = ChatOpenAI(
    model="qwen3-1.7b",              # здесь точное имя модели из LM Studio
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",             # любой строковый ключ, LM Studio его не проверяет
    temperature=0,
)


# 2) Подключение к Neo4j
graph = Neo4jGraph(
    url="bolt://localhost:7687",
    username="neo4j",
    password="airestairest",
)

# 3) Аналог sql_agent, но для Neo4j (GraphCypherQAChain)
cypher_qa = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    allow_dangerous_requests=False,   # лучше только READ-запросы в проде
)

ValueError: Could not use APOC procedures. Please ensure the APOC plugin is installed in Neo4j and that 'apoc.meta.data()' is allowed in Neo4j configuration 

In [ ]:
res = cypher_qa.invoke({"query": "Какие узлы и связи есть в графе?"})
print(res["result"])